In [1]:
%pip install -r requirements/requirements.txt
%pip install -r requirements/experiment_requirements.txt

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
if sys.platform == "darwin":
    # macOS
    %pip install jax-metal
    print('Installed jax-metal for macOS')
elif sys.platform == "linux":
    # Linux
    %pip install "jax[cuda12]"
    print('Installed jax[cuda12] for Linux')
else:
    print(f"Unsupported platform: {platform.system()}")

Note: you may need to restart the kernel to use updated packages.
Installed jax-metal for macOS


In [3]:
from experiment_helpers import (
    DatasetConfig,
    MPNNConfig,
    AggregationMode,
    run_experiment,
)
import jax

algorithm = "matrix_chain_order"

standard_dataset = DatasetConfig(
    algorithm_name=algorithm,
    num_samples=1000,
    length=16,
    train_batch_size=32,
    test_batch_size=8,
    num_test_samples=32,
)

In [4]:
print("Running on", jax.devices()[-1].device_kind)

Metal device set to: Apple M3 Pro

systemMemory: 18.00 GB
maxCacheSize: 6.00 GB

Running on Metal


W0000 00:00:1748014912.146135  422382 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1748014912.195947  422382 service.cc:145] XLA service 0x30c20c160 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748014912.195964  422382 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1748014912.197199  422382 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1748014912.197208  422382 mps_client.cc:384] XLA backend will use up to 12884443136 bytes on device 0 for SimpleAllocator.


In [5]:
default_learning_rate = 1e-3

In [ ]:
default_mpnn_config = MPNNConfig(
    aggregation_modes=[AggregationMode.SUM, AggregationMode.MAX],
    decoder_learning_rate=default_learning_rate,
    encoder_learning_rate=default_learning_rate,
    backbone_learning_rate=default_learning_rate,
    max_steps=6000,
    disable_jit=False,
    message_weight_decay=0.0,
    hint_teacher_forcing=0.0,
    hidden_dim=64,
    dropout_prob=0.0,
    nb_heads=4,
)
experiment_name = "Baseline test"
run_experiment(standard_dataset, default_mpnn_config, experiment_name=experiment_name)

Checking for sampler at samplers/sampler_matrix_chain_order_1000_16.data
Loaded sampler for algorithm: matrix_chain_order num_samples: 1000 length: 16
Checking for sampler at samplers/sampler_matrix_chain_order_32_64.data
Loaded sampler for algorithm: matrix_chain_order num_samples: 32 length: 64


wandb: Currently logged in as: jakosimov (jakosimov-university-of-cambridge) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


KeyboardInterrupt: 

In [ ]:
different_lr_mpnn_config = MPNNConfig(
    aggregation_modes=[AggregationMode.SUM, AggregationMode.MAX],
    decoder_learning_rate=default_learning_rate * 0.01,
    encoder_learning_rate=default_learning_rate,
    backbone_learning_rate=default_learning_rate * 0.1,
    max_steps=6000,
    disable_jit=False,
    message_weight_decay=0.0,
    hint_teacher_forcing=0.0,
    hidden_dim=64,
    dropout_prob=0.0,
    nb_heads=4,
)
experiment_name = "Different learning rates"
run_experiment(
    standard_dataset, different_lr_mpnn_config, experiment_name=experiment_name
)

In [ ]:
different_lr_decay_mpnn_config = MPNNConfig(
    aggregation_modes=[AggregationMode.SUM, AggregationMode.MAX],
    decoder_learning_rate=default_learning_rate * 0.01,
    encoder_learning_rate=default_learning_rate,
    backbone_learning_rate=default_learning_rate * 0.1,
    max_steps=6000,
    disable_jit=False,
    message_weight_decay=1e-4,
    hint_teacher_forcing=0.0,
    hidden_dim=64,
    dropout_prob=0.0,
    nb_heads=4,
)
experiment_name = "Different learning rates and message weight decay"
run_experiment(
    standard_dataset, different_lr_decay_mpnn_config, experiment_name=experiment_name
)